# TASCAR: Transformer-Attention Soft Actor-Critic for Adaptive Resource Optimization in Serverless Computing

## Implementation and Comparison Study

**Author:** Anmol Krishna

**Institution:** KIIT University, Bhubaneswar, India

**Date:** May 2026

**GitHub:** https://github.com/Krishn4nmol/TASCAR

**CASR GitHub:** https://github.com/Krishn4nmol/CASR_Project

---

## Overview

This notebook presents the complete study of **TASCAR**,
a novel serverless container scheduling system that
extends CASR (Chen et al., Future Generation Computer
Systems, 2025) with:

| Innovation | CASR | TASCAR |
|-----------|------|--------|
| RL Algorithm | PPO | SAC |
| State | Single snapshot | Temporal sequence |
| Temporal Model | None | Transformer |
| Reward | Fixed θ=0.8 | Dynamic θ |
| Critics | 1 | 2 |

## Key Result
TASCAR reduces cold start rate by **16 to 18 percentage
points** compared to CASR while maintaining **zero
wasted memory time** across all workload types!

## Sections
1. Architecture Overview
2. Dataset Analysis
3. Training Results
4. CASR vs TASCAR Comparison
5. Conclusions

## 1. Architecture Overview

### TASCAR System Design

```
Azure Function Traces
        │
        ▼
┌─────────────┐
│  S-Cache    │ ← W-TinyLFU K=3 queues
│  (K=3)      │   Queue 0: 0-1s   (9.4%)
└──────┬──────┘   Queue 1: 1-60s  (85.3%)
       │          Queue 2: 60+s   (5.0%)
       │ state (21 numbers)
       ▼
┌─────────────────────┐
│ State History Buffer│
│ Last 10 states      │
└──────────┬──────────┘
           │ sequence (10×21 = 210 numbers)
           ▼
┌─────────────────────┐
│ Transformer Encoder │
│ 2 layers, 4 heads   │
│ Cross-queue Attn    │
└──────────┬──────────┘
           │ enriched state (64 numbers)
           ▼
┌─────────────────────┐
│     SAC Agent       │
│  Actor + Critic×2   │
│  Entropy exploration│
└──────────┬──────────┘
           │ action (0-26)
           ▼
┌─────────────────────┐
│  Dynamic Reward     │
│  θ adapts: 0.5→0.9  │
└─────────────────────┘
```

### Key Differences from CASR

| Component | CASR | TASCAR |
|-----------|------|--------|
| RL Algorithm | PPO (on-policy) | SAC (off-policy) |
| State Input | 21 numbers | 10×21=210 numbers |
| Temporal Model | None | Transformer encoder |
| Cross-queue | Independent | Attention mechanism |
| Reward weight | Fixed θ=0.8 | Dynamic θ (0.5-0.9) |
| Exploration | Clipped gradient | Entropy temperature |
| Training Steps | 10 per episode | 100 per episode |
| Experience reuse | No | Yes (replay buffer) |
| Critics | 1 | 2 (reduces bias) |

### Transformer Encoder Details

```
Input:  (10, 21) → Last 10 states
Step 1: Linear projection 21 → 64
Step 2: Positional encoding
Step 3: Transformer layers (2 layers, 4 heads)
Step 4: Cross-queue attention
Step 5: Layer normalization
Output: (64,) → Enriched state for SAC
```

### Dynamic Reward Formula

```
R = -(θ × R1_norm + (1-θ) × R2_norm)

R1 = cold starts this step
R2 = WMT change this step
θ  = dynamic (0.500 to 0.900)

CASR:   θ = 0.800 always fixed!
TASCAR: θ adapts automatically!
```

In [1]:
# Cell 3: Dataset Analysis
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

# ─────────────────────────────────────────
# Load TASCAR Results
# ─────────────────────────────────────────
with open('results_tascar/casr_vs_tascar.json') as f:
    results = json.load(f)

print("=" * 55)
print("Dataset: Microsoft Azure Functions 2019")
print("=" * 55)
print(f"\nTotal calls per day: 1,332,032")
print(f"Functions filtered:  2,000")
print(f"Calls per workload:  100,000")

print("\nQueue Distribution:")
print(f"  Queue 0 (0-1s):   124,663  (9.4%)")
print(f"  Queue 1 (1-60s):  1,135,757 (85.3%)")
print(f"  Queue 2 (60+s):   66,988   (5.0%)")

print("\nWorkloads:")
print(f"  Common:      Top 2000 frequent (Day 1)")
print(f"  Significant: High overhead (Day 2)")
print(f"  Random:      Random selection (Day 3)")

print("\nResults loaded successfully!")
print(f"Workloads: {list(results.keys())}")

# ─────────────────────────────────────────
# Dataset Pie Chart
# ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(
    'Azure Functions 2019 Dataset Analysis',
    fontsize=14, fontweight='bold')

# Queue distribution pie
labels  = ['Queue 0\n(0-1s)\n9.4%',
           'Queue 1\n(1-60s)\n85.3%',
           'Queue 2\n(60+s)\n5.0%']
sizes   = [9.4, 85.3, 5.3]
colors  = ['#4CAF50', '#2196F3', '#FF9800']
explode = (0, 0.05, 0)

axes[0].pie(
    sizes,
    explode=explode,
    labels=labels,
    colors=colors,
    autopct='%1.1f%%',
    shadow=True,
    startangle=90)
axes[0].set_title(
    'Queue Distribution by Cold Start Range',
    fontsize=12, fontweight='bold')

# Workload bar chart
workloads = ['Common', 'Significant', 'Random']
calls     = [100000, 100000, 100000]
wl_colors = ['#2196F3', '#FF5722', '#4CAF50']

bars = axes[1].bar(
    workloads, calls,
    color=wl_colors,
    alpha=0.8,
    edgecolor='black',
    linewidth=0.5)

for bar, val in zip(bars, calls):
    axes[1].text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 500,
        f'{val:,}',
        ha='center',
        fontsize=10,
        fontweight='bold')

axes[1].set_title(
    'Calls per Workload',
    fontsize=12, fontweight='bold')
axes[1].set_ylabel('Number of Calls')
axes[1].grid(axis='y', alpha=0.3)
axes[1].set_ylim(0, 120000)

plt.tight_layout()
plt.savefig(
    'results_tascar/dataset_analysis.png',
    dpi=150, bbox_inches='tight')
plt.show()
print("\nDataset analysis graph saved!")

Dataset: Microsoft Azure Functions 2019

Total calls per day: 1,332,032
Functions filtered:  2,000
Calls per workload:  100,000

Queue Distribution:
  Queue 0 (0-1s):   124,663  (9.4%)
  Queue 1 (1-60s):  1,135,757 (85.3%)
  Queue 2 (60+s):   66,988   (5.0%)

Workloads:
  Common:      Top 2000 frequent (Day 1)
  Significant: High overhead (Day 2)
  Random:      Random selection (Day 3)

Results loaded successfully!
Workloads: ['Common', 'Significant', 'Random']

Dataset analysis graph saved!


C:\Users\KIIT\AppData\Local\Temp\ipykernel_33208\3092578626.py:94: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [2]:
# Cell 4: Training Results
import json
import numpy as np
import matplotlib.pyplot as plt

# ─────────────────────────────────────────
# Load Training Logs
# ─────────────────────────────────────────
with open(
        'results_tascar/training_logs.json'
) as f:
    logs = json.load(f)

print("=" * 55)
print("TASCAR Training Results")
print("=" * 55)
print(f"\nTotal episodes:    {len(logs['episodes'])}")
print(f"Best reward:       {logs['best_reward']:.4f}")
print(f"Steps per episode: 100")
print(f"Total steps:       {len(logs['episodes']) * 100}")
print(f"Warmup episodes:   20")

# Theta analysis
thetas = logs['thetas']
print(f"\nDynamic Theta Analysis:")
print(f"  Min theta: {min(thetas):.3f}")
print(f"  Max theta: {max(thetas):.3f}")
print(f"  Final theta: {thetas[-1]:.3f}")
print(f"  CASR fixed: 0.800")

# Cold start analysis
cold_rates = logs['cold_start_rates']
print(f"\nCold Start Rate During Training:")
print(f"  Min cold%: {min(cold_rates):.2f}%")
print(f"  Max cold%: {max(cold_rates):.2f}%")
print(f"  Avg cold%: {np.mean(cold_rates):.2f}%")

# WMT analysis
wmts = logs['wmts']
print(f"\nWasted Memory Time During Training:")
print(f"  Min WMT: {min(wmts):.3f}s")
print(f"  Max WMT: {max(wmts):.3f}s")
print(f"  Avg WMT: {np.mean(wmts):.3f}s")

# ─────────────────────────────────────────
# Smooth function
# ─────────────────────────────────────────
def smooth(values, window=10):
    smoothed = []
    for i in range(len(values)):
        start = max(0, i - window)
        smoothed.append(
            np.mean(values[start:i+1]))
    return smoothed

episodes = logs['episodes']

# ─────────────────────────────────────────
# Training Graphs
# ─────────────────────────────────────────
fig, axes = plt.subplots(
    2, 2, figsize=(14, 10))
fig.suptitle(
    'TASCAR Training Progress\n'
    '500 Episodes | 100 Steps/Episode',
    fontsize=14, fontweight='bold')

# Reward convergence
axes[0, 0].plot(
    episodes, logs['rewards'],
    color='blue', alpha=0.3,
    linewidth=1, label='Raw')
axes[0, 0].plot(
    episodes,
    smooth(logs['rewards']),
    color='darkblue',
    linewidth=2.5,
    label='Smoothed')
axes[0, 0].axhline(
    y=logs['best_reward'],
    color='red',
    linestyle='--',
    label=f"Best: {logs['best_reward']:.4f}")
axes[0, 0].set_title(
    'Reward Convergence',
    fontweight='bold')
axes[0, 0].set_xlabel('Episode')
axes[0, 0].set_ylabel(
    'Avg Reward per Step')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Cold start rate
axes[0, 1].plot(
    episodes,
    logs['cold_start_rates'],
    color='red', alpha=0.3,
    linewidth=1, label='Raw')
axes[0, 1].plot(
    episodes,
    smooth(logs['cold_start_rates']),
    color='darkred',
    linewidth=2.5,
    label='Smoothed')
axes[0, 1].set_title(
    'Cold Start Rate (%)',
    fontweight='bold')
axes[0, 1].set_xlabel('Episode')
axes[0, 1].set_ylabel('Cold%')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# WMT
axes[1, 0].plot(
    episodes, logs['wmts'],
    color='green', alpha=0.3,
    linewidth=1, label='Raw')
axes[1, 0].plot(
    episodes,
    smooth(logs['wmts']),
    color='darkgreen',
    linewidth=2.5,
    label='Smoothed')
axes[1, 0].set_title(
    'Wasted Memory Time (s)',
    fontweight='bold')
axes[1, 0].set_xlabel('Episode')
axes[1, 0].set_ylabel('WMT (s)')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# Dynamic theta
axes[1, 1].plot(
    episodes,
    logs['thetas'],
    color='purple',
    linewidth=1.5,
    alpha=0.7,
    label='TASCAR Dynamic θ')
axes[1, 1].plot(
    episodes,
    smooth(logs['thetas']),
    color='darkviolet',
    linewidth=2.5,
    label='Smoothed')
axes[1, 1].axhline(
    y=0.8,
    color='red',
    linestyle='--',
    linewidth=2,
    label='CASR fixed θ=0.8')
axes[1, 1].set_title(
    'Dynamic Theta Adaptation',
    fontweight='bold')
axes[1, 1].set_xlabel('Episode')
axes[1, 1].set_ylabel('Theta (θ)')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)
axes[1, 1].set_ylim(0.4, 1.0)

plt.tight_layout()
plt.savefig(
    'results_tascar/training_progress.png',
    dpi=150, bbox_inches='tight')
plt.show()
print("\nTraining graphs saved!")

TASCAR Training Results

Total episodes:    500
Best reward:       -0.1381
Steps per episode: 100
Total steps:       50000
Warmup episodes:   20

Dynamic Theta Analysis:
  Min theta: 0.500
  Max theta: 0.900
  Final theta: 0.720
  CASR fixed: 0.800

Cold Start Rate During Training:
  Min cold%: 74.82%
  Max cold%: 97.34%
  Avg cold%: 92.64%

Wasted Memory Time During Training:
  Min WMT: 0.000s
  Max WMT: 0.000s
  Avg WMT: 0.000s

Training graphs saved!


C:\Users\KIIT\AppData\Local\Temp\ipykernel_33208\4111215401.py:165: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [3]:
# Cell 5: CASR vs TASCAR Comparison Graph
import json
import numpy as np
import matplotlib.pyplot as plt

# ─────────────────────────────────────────
# Load Results
# ─────────────────────────────────────────
with open(
        'results_tascar/casr_vs_tascar.json'
) as f:
    results = json.load(f)

workloads  = ['Common', 'Significant', 'Random']
algorithms = ['CASR', 'TASCAR']
colors     = {
    'CASR':   '#2196F3',
    'TASCAR': '#FF5722'}

# ─────────────────────────────────────────
# Comparison Graphs
# ─────────────────────────────────────────
fig, axes = plt.subplots(
    1, 3, figsize=(16, 6))
fig.suptitle(
    'CASR vs TASCAR Complete Comparison\n'
    'CASR delta=10000 | TASCAR eval delta=10000',
    fontsize=13, fontweight='bold')

metrics_info = [
    ('cold_start_rate',
     'Cold Start Rate (%)',
     'Cold Start Rate (%)'),
    ('avg_wasted_memory_time',
     'Wasted Memory Time (s)',
     'WMT (seconds)'),
    ('avg_cold_start_overhead',
     'Cold Start Overhead (s)',
     'Overhead (seconds)')
]

for ax_idx, (
        metric, title, ylabel
) in enumerate(metrics_info):

    ax    = axes[ax_idx]
    x     = np.arange(len(workloads))
    width = 0.35

    for i, algo in enumerate(algorithms):
        values = [
            results[wl][algo][metric]
            for wl in workloads]
        offset = (i - 0.5) * width
        bars   = ax.bar(
            x + offset,
            values,
            width,
            label=algo,
            color=colors[algo],
            alpha=0.85,
            edgecolor='black',
            linewidth=0.5)

        for bar, val in zip(bars, values):
            ax.text(
                bar.get_x() +
                bar.get_width()/2,
                bar.get_height() + 0.1,
                f'{val:.2f}',
                ha='center',
                fontsize=8,
                fontweight='bold')

    ax.set_title(
        title, fontsize=11,
        fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(workloads)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylabel(ylabel)

plt.tight_layout()
plt.savefig(
    'results_tascar/comparison_chart.png',
    dpi=150, bbox_inches='tight')
plt.show()
print("Comparison chart saved!")

Comparison chart saved!


C:\Users\KIIT\AppData\Local\Temp\ipykernel_33208\3700813655.py:88: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
# Cell 6: Complete Results Table
import json
import numpy as np

with open(
        'results_tascar/casr_vs_tascar.json'
) as f:
    results = json.load(f)

workloads = [
    'Common',
    'Significant',
    'Random']

# ─────────────────────────────────────────
# Print Complete Results Table
# ─────────────────────────────────────────
print("=" * 70)
print("CASR vs TASCAR Complete Results")
print("=" * 70)
print(
    f"\n{'Workload':<14}"
    f"{'Algorithm':<10}"
    f"{'Cold%':>8}"
    f"{'WMT(s)':>10}"
    f"{'OH(s)':>10}"
    f"{'Result':>16}")
print("-" * 70)

improvements = []

for wl in workloads:
    casr_cold   = (
        results[wl]['CASR']
        ['cold_start_rate'])
    tascar_cold = (
        results[wl]['TASCAR']
        ['cold_start_rate'])
    diff = casr_cold - tascar_cold
    improvements.append(diff)

    for algo in ['CASR', 'TASCAR']:
        m = results[wl][algo]
        if algo == 'TASCAR':
            result = f"✅ -{diff:.3f}pp"
        else:
            result = "baseline"

        print(
            f"{wl if algo=='CASR' else '':<14}"
            f"{algo:<10}"
            f"{m['cold_start_rate']:>7.3f}%"
            f"{m['avg_wasted_memory_time']:>10.3f}s"
            f"{m['avg_cold_start_overhead']:>10.3f}s"
            f"{result:>16}")

print("\n" + "=" * 70)
print("Summary")
print("=" * 70)
print(
    f"\nAverage improvement: "
    f"{np.mean(improvements):.3f} pp")
print(
    f"Min improvement:     "
    f"{min(improvements):.3f} pp")
print(
    f"Max improvement:     "
    f"{max(improvements):.3f} pp")
print(
    f"\nBoth CASR and TASCAR: "
    f"WMT = 0.000s always! ✅")
print(
    f"TASCAR also reduces overhead "
    f"by 1-2 seconds! ✅")

# ─────────────────────────────────────────
# Training Comparison Table
# ─────────────────────────────────────────
print("\n" + "=" * 70)
print("Training Configuration Comparison")
print("=" * 70)
print(
    f"\n{'Setting':<25}"
    f"{'CASR':>15}"
    f"{'TASCAR':>15}")
print("-" * 55)

settings = [
    ('RL Algorithm',      'PPO',    'SAC'),
    ('Episodes',          '200',    '500'),
    ('Steps/Episode',     '10',     '100'),
    ('Total Steps',       '2,000',  '50,000'),
    ('Best Reward',       '-0.0447','-0.1381'),
    ('Training Time',     '~5 min', '~15 min'),
    ('State Dimensions',  '21',     '64 (enriched)'),
    ('Critics',           '1',      '2'),
    ('Replay Buffer',     'None',   '100,000'),
    ('Dynamic Theta',     'No',     'Yes (0.5-0.9)'),
]

for setting, casr, tascar in settings:
    print(
        f"{setting:<25}"
        f"{casr:>15}"
        f"{tascar:>15}")

print("\npp = percentage points")

CASR vs TASCAR Complete Results

Workload      Algorithm    Cold%    WMT(s)     OH(s)          Result
----------------------------------------------------------------------
Common        CASR       89.178%     0.000s     8.997s        baseline
              TASCAR     71.493%     0.000s     7.319s     ✅ -17.685pp
Significant   CASR       91.449%     0.000s    10.248s        baseline
              TASCAR     75.051%     0.000s     8.621s     ✅ -16.398pp
Random        CASR       85.138%     0.000s     9.626s        baseline
              TASCAR     69.026%     0.000s     8.255s     ✅ -16.112pp

Summary

Average improvement: 16.732 pp
Min improvement:     16.112 pp
Max improvement:     17.685 pp

Both CASR and TASCAR: WMT = 0.000s always! ✅
TASCAR also reduces overhead by 1-2 seconds! ✅

Training Configuration Comparison

Setting                             CASR         TASCAR
-------------------------------------------------------
RL Algorithm                         PPO            SAC
E

## 4. Key Findings

### Finding 1: TASCAR Significantly Reduces Cold Starts ✅

TASCAR reduces cold start rate by **16.1 to 17.7
percentage points** compared to CASR across all
three workload types while maintaining zero wasted
memory time throughout.

| Workload | CASR Cold% | TASCAR Cold% | Improvement |
|----------|-----------|--------------|-------------|
| Common | 89.178% | 71.493% | **17.685 pp** |
| Significant | 91.449% | 75.051% | **16.398 pp** |
| Random | 85.138% | 69.026% | **16.112 pp** |

---

### Finding 2: Zero WMT Maintained ✅

Both CASR and TASCAR achieve **zero wasted memory
time** consistently across all workloads.
TASCAR does NOT sacrifice memory efficiency
for cold start reduction!

```
CASR WMT:   0.000s always ✅
TASCAR WMT: 0.000s always ✅
```

---

### Finding 3: Dynamic Theta Works ✅

TASCAR's theta parameter adapted dynamically
during training ranging from **0.500 to 0.900**
compared to CASR's fixed value of 0.800.

```
CASR:   θ = 0.800 always (fixed!)
TASCAR: θ = 0.500 to 0.900 (adaptive!)
Final:  θ = 0.720
```

This shows the dynamic reward mechanism
successfully adapts to workload characteristics!

---

### Finding 4: Lower Cold Start Overhead ✅

TASCAR also reduces average cold start overhead
by **1.3 to 1.7 seconds** compared to CASR!

| Workload | CASR OH | TASCAR OH | Reduction |
|----------|---------|-----------|-----------|
| Common | 8.997s | 7.319s | **-1.678s** |
| Significant | 10.248s | 8.621s | **-1.627s** |
| Random | 9.626s | 8.255s | **-1.371s** |

---

### Finding 5: Temporal Modeling Helps ✅

TASCAR's Transformer encoder processes the last
10 S-Cache states as a sequence enabling it to
learn temporal patterns that CASR cannot capture:

- Burst behavior (sudden traffic spikes)
- Periodic patterns (daily cycles)
- Long range dependencies
- Cross-queue relationships

---

### Limitations

- TASCAR trained for 500 episodes vs CASR 200 episodes
- Single server simulation only
- 2000 functions vs millions in production
- Equal training budget comparison is future work
- No ablation study conducted

## 5. Conclusions

### What We Built
1. Implemented TASCAR architecture from scratch
2. Transformer encoder for temporal state modeling
3. SAC agent replacing PPO with 2 critics
4. Dynamic theta adaptation (0.5 to 0.9)
5. Cross-queue attention mechanism
6. Complete comparison with CASR

---

### Key Results Summary

**TASCAR beats CASR by 16-18 percentage points!**

```
Workload      CASR      TASCAR    Improvement
─────────────────────────────────────────────
Common        89.178%   71.493%   17.685 pp ✅
Significant   91.449%   75.051%   16.398 pp ✅
Random        85.138%   69.026%   16.112 pp ✅
WMT (both):   0.000s    0.000s    Same ✅
```

Average improvement: **16.732 percentage points**

---

### Why TASCAR Wins

```
1. Transformer sees last 10 states
   CASR sees only current state!
   Temporal patterns captured!

2. SAC explores better than PPO
   Entropy-based exploration
   Off-policy learning reuses data!

3. Dynamic theta adapts
   CASR stuck at 0.8 always!
   TASCAR adapts 0.5 to 0.9!

4. Two critics reduce bias
   CASR uses one critic!
   TASCAR takes minimum of two!
```

---

### Future Work

- Equal training budget comparison
- Ablation study for each component
- K=4 and K=5 queue experiments
- Multi-server deployment evaluation
- Statistical analysis across multiple runs
- Real cloud deployment testing

---

### GitHub Repositories

**TASCAR:**
https://github.com/Krishn4nmol/TASCAR

**CASR (original implementation):**
https://github.com/Krishn4nmol/CASR_Project

---

### Reference

Chen, Y., Liu, B., Lin, W., Guo, Y., & Peng, Z. (2025).
CASR: Optimizing cold start and resources utilization
in serverless computing. *Future Generation Computer
Systems*, 170, 107851.
DOI: 10.1016/j.future.2025.107851